# 03 - Huấn luyện mô hình

Notebook này huấn luyện Isolation Forest để phát hiện bất thường và Random Forest để phân loại bot.

## Bước 0 - Cấu hình đường dẫn local hoặc Google Colab

Nếu chạy trên Colab, hãy đặt project tại `/content/drive/MyDrive/bot-detection-project`.

In [1]:
from pathlib import Path
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/bot-detection-project')
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == 'notebooks':
        PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

Project root: d:\Data mining\bot-detection-project


In [2]:
if IN_COLAB:
    %pip install -q -r "{PROJECT_ROOT / 'requirements.txt'}"

## Bước 1 - Tải dữ liệu đặc trưng

In [3]:
import json
import joblib
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from src.models import run_isolation_forest, run_random_forest

SEED = 42
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'outputs' / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

features_df = pd.read_csv(PROCESSED_DIR / 'features.csv')
X = features_df.drop(columns=['label'])
y = features_df['label'].astype(int)
print(f'Đã tải dữ liệu: X={X.shape}, y={y.shape}')

Đã tải dữ liệu: X=(10514, 20), y=(10514,)


## Bước 2 - Chia train/test và xử lý dữ liệu thiếu

In [4]:
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y,
)

# 1. Log transform skewed features for train and test
X_train_log = X_train.copy()
X_test_log = X_test.copy()
skewed_cols = ["followers_count", "friends_count", "statuses_count", "favourites_count", "followers_friends_ratio"]
for col in skewed_cols:
    if col in X_train_log.columns:
        X_train_log[col] = np.log1p(X_train_log[col].clip(lower=0))
        X_test_log[col] = np.log1p(X_test_log[col].clip(lower=0))

# 2. Impute and Scale for Random Forest (all 20 features)
imputer = SimpleImputer(strategy='median', keep_empty_features=True)
X_train_imputed = pd.DataFrame(imputer.fit_transform(X_train_log), columns=X.columns, index=X_train.index)
X_test_imputed = pd.DataFrame(imputer.transform(X_test_log), columns=X.columns, index=X_test.index)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_imputed), columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_imputed), columns=X.columns, index=X_test.index)

# 3. Impute and Scale for Isolation Forest (7 continuous features only)
iso_features = ["followers_count", "friends_count", "followers_friends_ratio", "statuses_count", "favourites_count", "account_age_days", "tweets_per_day"]
X_train_iso = X_train_log[iso_features].copy()
X_test_iso = X_test_log[iso_features].copy()

imputer_iso = SimpleImputer(strategy='median', keep_empty_features=True)
X_train_imputed_iso = pd.DataFrame(imputer_iso.fit_transform(X_train_iso), columns=iso_features, index=X_train.index)
X_test_imputed_iso = pd.DataFrame(imputer_iso.transform(X_test_iso), columns=iso_features, index=X_test.index)

scaler_iso = StandardScaler()
X_train_scaled_iso = pd.DataFrame(scaler_iso.fit_transform(X_train_imputed_iso), columns=iso_features, index=X_train.index)
X_test_scaled_iso = pd.DataFrame(scaler_iso.transform(X_test_imputed_iso), columns=iso_features, index=X_test.index)

print(f'Tập train RF: {X_train_scaled.shape}')
print(f'Tập test RF: {X_test_scaled.shape}')
print(f'Tập train ISO: {X_train_scaled_iso.shape}')
print(f'Tập test ISO: {X_test_scaled_iso.shape}')
print('Phân phối nhãn train:')
print(y_train.value_counts().rename(index={0: 'người thật', 1: 'bot'}))

Tập train RF: (8411, 20)
Tập test RF: (2103, 20)
Tập train ISO: (8411, 7)
Tập test ISO: (2103, 7)
Phân phối nhãn train:
label
bot           5632
người thật    2779
Name: count, dtype: int64


## Bước 3 - Chọn ngưỡng cảnh báo và huấn luyện hai mô hình

Isolation Forest chỉ học từ tài khoản người thật thuộc tập train. Một tập validation nhỏ được dùng để chọn contamination cân bằng hơn giữa precision và recall. Random Forest học từ toàn bộ tập train có nhãn.

In [5]:
X_iso_fit, X_iso_val, y_iso_fit, y_iso_val = train_test_split(
    X_train_scaled_iso, y_train, test_size=0.20, random_state=SEED, stratify=y_train,
)
candidate_contaminations = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
validation_rows = []
for contamination in candidate_contaminations:
    validation_result = run_isolation_forest(
        X_iso_fit.loc[y_iso_fit == 0], X_iso_val, y_iso_val, contamination=contamination,
    )
    validation_rows.append({'contamination': contamination, 'f1': validation_result['f1']})

validation_df = pd.DataFrame(validation_rows)
best_contamination = float(validation_df.loc[validation_df['f1'].idxmax(), 'contamination'])
print(f'Contamination được chọn từ validation: {best_contamination:.2f}')
display(validation_df)

X_human_train_iso = X_train_scaled_iso.loc[y_train == 0]
iso_results = run_isolation_forest(X_human_train_iso, X_test_scaled_iso, y_test, contamination=best_contamination)
rf_results = run_random_forest(X_train_scaled, X_test_scaled, y_train, y_test)

results = {
    'Isolation Forest': iso_results,
    'Random Forest': rf_results,
}

[mô hình] Đang huấn luyện Isolation Forest trên tài khoản người thật...


[mô hình] Hoàn tất Isolation Forest
[mô hình] Đang huấn luyện Isolation Forest trên tài khoản người thật...


[mô hình] Hoàn tất Isolation Forest
[mô hình] Đang huấn luyện Isolation Forest trên tài khoản người thật...


[mô hình] Hoàn tất Isolation Forest
[mô hình] Đang huấn luyện Isolation Forest trên tài khoản người thật...


[mô hình] Hoàn tất Isolation Forest
[mô hình] Đang huấn luyện Isolation Forest trên tài khoản người thật...


...

[mô hình] Hoàn tất Isolation Forest
Contamination được chọn từ validation: 0.25


,contamination,f1
0,0.05,0.868588
1,0.10,0.911765
2,0.15,0.930439
3,0.20,0.939250
4,0.25,0.944047
5,0.30,0.931234
6,0.35,0.923583
7,0.40,0.913079
8,0.45,0.899200
9,0.50,0.891003


[mô hình] Đang huấn luyện Isolation Forest trên tài khoản người thật...


[mô hình] Hoàn tất Isolation Forest
[mô hình] Đang huấn luyện Random Forest...


[mô hình] Hoàn tất Random Forest


## Bước 4 - Hiển thị kết quả

In [6]:
summary_df = pd.DataFrame([
    {
        'Mô hình': model_name,
        'Accuracy': values['accuracy'],
        'Precision': values['precision'],
        'Recall': values['recall'],
        'F1': values['f1'],
        'AUC': values['roc_auc'],
    }
    for model_name, values in results.items()
])
display(summary_df.round(4))

for model_name, values in results.items():
    print('\n' + '=' * 70)
    print(model_name)
    print('Ma trận nhầm lẫn:')
    print(values['confusion_matrix'])
    print(values['classification_report'])

,Mô hình,Accuracy,Precision,Recall,F1,AUC
0,Isolation Forest,0.9163,0.8944,0.9922,0.9407,0.9603
1,Random Forest,0.9872,0.9964,0.9844,0.9904,0.9965



Isolation Forest
Ma trận nhầm lẫn:
[[ 530  165]
 [  11 1397]]
              precision    recall  f1-score   support

       human       0.98      0.76      0.86       695
         bot       0.89      0.99      0.94      1408

    accuracy                           0.92      2103
   macro avg       0.94      0.88      0.90      2103
weighted avg       0.92      0.92      0.91      2103


Random Forest
Ma trận nhầm lẫn:
[[ 690    5]
 [  22 1386]]
              precision    recall  f1-score   support

       human       0.97      0.99      0.98       695
         bot       1.00      0.98      0.99      1408

    accuracy                           0.99      2103
   macro avg       0.98      0.99      0.99      2103
weighted avg       0.99      0.99      0.99      2103



## Bước 5 - Lưu mô hình, dữ liệu test và metrics

In [7]:
joblib.dump(imputer, MODELS_DIR / 'imputer.pkl')
joblib.dump(scaler, MODELS_DIR / 'scaler.pkl')
joblib.dump(imputer_iso, MODELS_DIR / 'imputer_iso.pkl')
joblib.dump(scaler_iso, MODELS_DIR / 'scaler_iso.pkl')
joblib.dump(iso_results['model'], MODELS_DIR / 'isolation_forest.pkl')
joblib.dump(rf_results['model'], MODELS_DIR / 'random_forest.pkl')

train_data = X_train.copy()
train_data['label'] = y_train.to_numpy()
train_data.to_csv(PROCESSED_DIR / 'train_data.csv', index=False)

test_data = X_test.copy()
test_data['label'] = y_test.to_numpy()
test_data.to_csv(PROCESSED_DIR / 'test_data.csv', index=False)

metrics = {
    'isolation_forest': {
        'name': 'Isolation Forest',
        'accuracy': round(iso_results['accuracy'], 4),
        'precision': round(iso_results['precision'], 4),
        'recall': round(iso_results['recall'], 4),
        'f1': round(iso_results['f1'], 4),
        'auc': round(iso_results['roc_auc'], 4),
        'contamination': best_contamination,
    },
    'random_forest': {
        'name': 'Random Forest',
        'accuracy': round(rf_results['accuracy'], 4),
        'precision': round(rf_results['precision'], 4),
        'recall': round(rf_results['recall'], 4),
        'f1': round(rf_results['f1'], 4),
        'auc': round(rf_results['roc_auc'], 4),
    },
}
with (MODELS_DIR / 'metrics.json').open('w', encoding='utf-8') as file:
    json.dump(metrics, file, ensure_ascii=False, indent=2)

print(f'Đã lưu mô hình và metrics tại: {MODELS_DIR}')

Đã lưu mô hình và metrics tại: d:\Data mining\bot-detection-project\outputs\models
